In [4]:

import json
import time

import psycopg
from psycopg import OperationalError


In [5]:
def load_data(filename):
    """
    Loads cleaned data from a JSON file.
    """
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

In [6]:
directory = '/Users/jennifer/Documents/software_concept_python_class/jhu_software_concepts/Module_2/'

applicant_data= load_data(directory+ "applicant_data_v2.json")
len(applicant_data)

41000

In [7]:
print(applicant_data[0].keys()) 

dict_keys(['Program Name', 'University', 'Comments', 'Date of Information Added to Grad Cafe', 'URL link to applicant entry', 'Applicant Status', 'Accepted: Acceptance Date', 'Rejected: Rejection Date', 'Semester and Year of Program Start', 'International / American Student', 'GRE Score', 'GRE V Score', 'Masters or PhD', 'GPA', 'GRE AW', 'extra_info', 'raw_texts', 'page_number', 'llm-generated-program', 'llm-generated-university'])


In [9]:
# Create connection to system database
def create_system_connection(user, password):
    """Connect to postgres system database"""
    try:
        conn = psycopg.connect(
            dbname="postgres",
            user=user,
            password=password,
            host="localhost"
        )
        conn.autocommit = True
        return conn
    except OperationalError as e:
        print(f"Connection failed: {e}")
        return None

In [11]:
system_conn = create_system_connection("postgres", "181818")

In [12]:
# Create new database
def create_database(conn, db_name):
    """Create a new database"""
    try:
        cur = conn.cursor()
        cur.execute(f"CREATE DATABASE {db_name};")
        cur.close()
        print(f"Database '{db_name}' created successfully")
    except OperationalError as e:
        print(f"Database creation failed: {e}")

create_database(system_conn, "gradcafe_db")
system_conn.close()

Database 'gradcafe_db' created successfully


In [ ]:
def create_db_connection(db_name, db_user, db_password, db_host, db_port):

    """
    create conenction to my db
    """

    connection = None
    try:
        connection = psycopg.connect(
            dbname=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
        )
        print("Connection to PostgreSQL DB successful")
    except OperationalError as e:
        print(f"The error '{e}' occurred")
    return connection

In [23]:
connection = create_db_connection("gradcafe_db", "postgres", "181818", "127.0.0.1", "5432")

Connection to PostgreSQL DB successful


In [25]:
def execute_query(connection, query):
    connection.autocommit = True
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        print("Query executed successfully")
    except OperationalError as e:
        print(f"The error '{e}' occurred")

#### create table

In [26]:
create_users_table = """
CREATE TABLE IF NOT EXISTS applicants (
p_id SERIAL PRIMARY KEY,
program	TEXT,
comments TEXT,
date_added date,
url	TEXT,
status TEXT,
term TEXT,
us_or_international	TEXT,
gpa	FLOAT,
gre	FLOAT,
gre_v FLOAT,
gre_aw FLOAT,
degree TEXT,
llm_generated_program TEXT,
llm_generated_university TEXT
)
"""

execute_query(connection, create_users_table)

Query executed successfully


#### insert records